# 06 — BodyM dataset exploration

**Research question:** Các tập BodyM có nguồn nhãn, phân bố và tính độc lập như thế nào?

Phase 2 của Module 01. Seed 42; dùng dữ liệu đã tải, không đổi split và không học từ test.

In [1]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Module_01_User_and_Body_Profile.md").exists())
for path in [ROOT, ROOT / "scripts"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from IPython.display import display, Markdown
import numpy as np
import matplotlib.pyplot as plt


In [2]:
manifest = json.loads((ROOT / "data/manifests/bodym_phase2_v1.json").read_text(encoding="utf-8"))
print("Images verified:", manifest["images_verified"])
print("Label provenance:", manifest["label_provenance"])
print("License:", manifest["license_status"])
display(manifest["splits"])
display(manifest["overlaps"])

Images verified: 17956
Label provenance: 14 circumferences/lengths computed from registered 3D-scan meshes, not tape/InBody labels
License: Registry/paper name BY-NC 4.0 but link to BY 4.0. Non-commercial research only pending clarification.


{'train': {'subjects': 2018, 'photo_pairs': 6134},
 'testA': {'subjects': 87, 'photo_pairs': 1684},
 'testB': {'subjects': 400, 'photo_pairs': 1160}}

{'train__testA': {'subject_ids': 0,
  'photo_ids': 0,
  'exact_image_content_hashes': 0},
 'train__testB': {'subject_ids': 0,
  'photo_ids': 0,
  'exact_image_content_hashes': 0},
 'testA__testB': {'subject_ids': 1,
  'photo_ids': 0,
  'exact_image_content_hashes': 0}}

## Dataset / baseline / experiment config

BodyM cung cấp silhouette trước/nghiêng, metadata và số đo từ 3D scan/mesh. Không có ảnh RGB, keypoint, phiếu InBody hoặc nhãn mỡ/cơ. Baseline Ridge 393 features và CNN hai view đã có; phase này giữ checkpoint và bổ sung kiểm tra. Registry ghi tên BY-NC nhưng link BY: chỉ nghiên cứu phi thương mại khi chưa làm rõ.

In [3]:
from run_bodym_baseline import read_csv
record = json.loads((ROOT / "configs/bodym_subject_split.json").read_text(encoding="utf-8"))
fit, val = set(record["fit_subject_ids"]), set(record["validation_subject_ids"])
assert not fit & val
tables = {split: read_csv(ROOT / "data/raw/bodym" / split / "hwg_metadata.csv") for split in ("train", "testA", "testB")}
assert fit | val == {r["subject_id"] for r in tables["train"]}
for split in ("testA", "testB"):
    assert not (fit | val) & {r["subject_id"] for r in tables[split]}
print("FIT:", len(fit), "Validation:", len(val), "Train/test subject isolation passed")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for split, rows in tables.items():
    heights = np.array([float(r["height_cm"]) for r in rows])
    weights = np.array([float(r["weight_kg"]) for r in rows])
    axes[0].hist(heights, bins=np.linspace(130, 210, 25), density=True, histtype="step", label=split)
    axes[1].hist(weights / (heights / 100) ** 2, bins=np.linspace(10, 60, 26), density=True, histtype="step", label=split)
axes[0].set_xlabel("Provided height (cm)"); axes[1].set_xlabel("BMI from provided metadata")
for ax in axes:
    ax.set_ylabel("Density, one row/person"); ax.legend()
fig.tight_layout(); plt.show()

FIT: 1614 Validation: 404 Train/test subject isolation passed


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8720\1190434097.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  fig.tight_layout(); plt.show()


## Metrics / error analysis / conclusion

2.505 dòng người qua ba tập nhưng 2.504 ID duy nhất: testA/testB chung 1 ID, không gộp như hai mẫu người độc lập. Toàn bộ ảnh/CSV đã kiểm ETag/size, manifest lưu hash. Histogram chỉ mô tả dữ liệu, không dùng chọn model/ngưỡng. Nhóm BMI ở notebook đánh giá lấy quartile từ FIT, không mang ý nghĩa ngưỡng sức khỏe. Nguồn: [AWS](https://registry.opendata.aws/bodym/), [bài báo](https://adversarialbodysim.github.io/).